In [34]:
from Base_Modules.Environments import Prison
from Prison_Strategies.Basic_Strategies import *
from Base_Modules.Game_Master import Game_Master
from Base_Modules.Action import Action_History, Prison_Actions, Duel_Matrix
import pandas as pd

In [35]:
prison = Prison()
actions = prison.Get_Actions()

strategies = {
    0: Random_Strategy(0, actions),
    1: Always_Cooperate(1, actions),
    2: Always_Betray(2, actions),
    3: Random_Strategy(3, actions, p_coop=0.1),
    4 : Patient_Unforgiving(4, actions),
}

In [36]:
gm = Game_Master(prison, strategies=strategies, duel_size=2)
duel_matrix, rewards = gm.Tournament(4, Game_Master.Game_Type.All_Vs_All, True)
rewards.Sort_Total_Rewards()

In [37]:
total_rewards = rewards.Get_Total_Rewards()
rewards_per_name = {str(strategies[i]):total_rewards[i] for i in strategies.keys()}
rewards_per_name = pd.DataFrame(rewards_per_name, index=strategies.keys())

duel_rewards = rewards.Get_All_Duel_Rewards()
winner = next(iter(total_rewards))

In [38]:
strat_names = [str(s) for s in strategies.values()]

# Initialize matrix with NaN
df = pd.DataFrame(index=strat_names, columns=strat_names, dtype=object)

for (p1, p2), scores in duel_rewards.items():
    df.loc[str(strategies[p2]), str(strategies[p1])] = (scores[p2], scores[p1])
    df.loc[str(strategies[p1]), str(strategies[p2])] = (scores[p1], scores[p2])

for p in strat_names:
    df.loc[p, p] = (0, 0)

In [39]:
print(type(df))

<class 'pandas.DataFrame'>


In [40]:
binary_df = df.apply(lambda col: col.map(lambda x: int(x[0] > x[1])))
for p in strat_names:
    binary_df.loc[p, p] = float("NaN")

print(binary_df)

                                  Random_Strategy (p_coop=0.5)  \
Random_Strategy (p_coop=0.5)                               NaN   
Always_Cooperate                                           0.0   
Always_Betray                                              1.0   
Random_Strategy (p_coop=0.1)                               1.0   
Patient_Unforgiving (patience=1)                           1.0   

                                  Always_Cooperate  Always_Betray  \
Random_Strategy (p_coop=0.5)                   1.0            0.0   
Always_Cooperate                               NaN            0.0   
Always_Betray                                  1.0            NaN   
Random_Strategy (p_coop=0.1)                   1.0            0.0   
Patient_Unforgiving (patience=1)               0.0            0.0   

                                  Random_Strategy (p_coop=0.1)  \
Random_Strategy (p_coop=0.5)                               0.0   
Always_Cooperate                                        